In [ ]:
#!/usr/bin/env python3
# Search for all trained model files in Google Drive

import os
import glob
from datetime import datetime

print("🔍 Searching for trained models in Google Drive...")
print("This may take 30-60 seconds...\n")

# Mount drive if not already mounted
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except:
    pass

# Search patterns
search_locations = [
    "/content/drive/MyDrive/**/*.pt",
    "/content/traffic_signs_work/**/*.pt",
    "/content/runs/**/*.pt",
]

all_models = []

for pattern in search_locations:
    found = glob.glob(pattern, recursive=True)
    all_models.extend(found)

# Remove duplicates
all_models = list(set(all_models))

if all_models:
    print(f"✅ Found {len(all_models)} model file(s):\n")
    
    # Sort by modification time (newest first)
    model_info = []
    for model_path in all_models:
        try:
            size_mb = os.path.getsize(model_path) / (1024 * 1024)
            mod_time = os.path.getmtime(model_path)
            mod_date = datetime.fromtimestamp(mod_time).strftime('%Y-%m-%d %H:%M')
            model_info.append({
                'path': model_path,
                'size': size_mb,
                'date': mod_date,
                'time': mod_time
            })
        except:
            pass
    
    # Sort by date (newest first)
    model_info.sort(key=lambda x: x['time'], reverse=True)
    
    for i, info in enumerate(model_info, 1):
        print(f"{i}. {info['path']}")
        print(f"   Size: {info['size']:.2f} MB | Modified: {info['date']}")
        print()
    
    # Show recommendation
    print("=" * 70)
    print("📝 RECOMMENDATIONS:")
    print("=" * 70)
    
    # Find best.pt files
    best_models = [m for m in model_info if 'best.pt' in m['path']]
    if best_models:
        print(f"\n✅ Found {len(best_models)} 'best.pt' model(s) (recommended):")
        for i, m in enumerate(best_models[:3], 1):  # Show top 3
            print(f"   {i}. {m['path']}")
    
    # Find traffic sign models
    traffic_models = [m for m in model_info if 'traffic' in m['path'].lower()]
    if traffic_models:
        print(f"\n🚦 Found {len(traffic_models)} traffic-related model(s):")
        for i, m in enumerate(traffic_models[:3], 1):
            print(f"   {i}. {m['path']}")
    
    print("\n" + "=" * 70)
    print("📱 NEXT STEPS:")
    print("=" * 70)
    print("1. Copy the path of the model you want to use")
    print("2. Run cell 13 (Step 11A) - it will automatically find and use it")
    print("   OR manually upload best.pt to: /content/drive/MyDrive/RoadAI_Models/")
    print("\n💡 TIP: Use the NEWEST 'best.pt' file for best results")
    
else:
    print("❌ No trained models found in Google Drive")
    print("\n📝 OPTIONS:")
    print("1. Train a new model: Run cells 1-10")
    print("2. Upload existing best.pt to: /content/drive/MyDrive/RoadAI_Models/")
    print("3. Check if you trained models in a different Google account")
    print("\n💡 If you trained before but can't find the model:")
    print("   - Check Google Drive trash/bin")
    print("   - Search Google Drive web interface for 'best.pt'")
    print("   - Verify you're using the same Google account")

## 🔍 Find Existing Trained Models

**Run this cell to search your Google Drive for any trained model files (.pt)**

This will help you locate models from previous training sessions.

In [ ]:
# Keep-alive script to prevent Colab from disconnecting
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   console.log("Attempting to keep session alive"); 
   document.querySelector("colab-connect-button").click()
 }
 setInterval(ClickConnect, 60000)
'''))

print("✓ Keep-alive script activated")

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully")

## Step 2: Install Required Libraries

In [ ]:
# Install YOLOv8 and dependencies
!pip install -q ultralytics opencv-python-headless pillow
print("✓ Libraries installed")

## Step 3: Set Up Paths

**IMPORTANT**: Update the `DATASET_PATH` below to point to your dataset folder in Google Drive!

In [ ]:
import os

# ⚠️ CHANGE THIS PATH to your dataset location in Google Drive
DATASET_PATH = "/content/drive/MyDrive/traffic_signs_mini"

# Verify path exists
if not os.path.exists(DATASET_PATH):
    print(f"❌ ERROR: Dataset path not found: {DATASET_PATH}")
    print("Please upload your dataset to Google Drive and update DATASET_PATH")
else:
    print(f"✓ Dataset found at: {DATASET_PATH}")
    
# Create working directory
WORK_DIR = "/content/traffic_signs_work"
os.makedirs(WORK_DIR, exist_ok=True)
print(f"✓ Working directory: {WORK_DIR}")

## Step 4: Define Class Names

**IMPORTANT**: Update `CLASS_NAMES` list to match your traffic sign classes!

In [ ]:
# ⚠️ UPDATE THIS LIST with your actual traffic sign classes
# Example: For demo with 5 sign types
CLASS_NAMES = [
    "Stop Sign",
    "Speed Limit 50",
    "Speed Limit 80",
    "No Entry",
    "Yield"
]

NUM_CLASSES = len(CLASS_NAMES)
print(f"✓ Number of classes: {NUM_CLASSES}")
print(f"✓ Classes: {CLASS_NAMES}")

## Step 5: Create YOLO Configuration File

In [ ]:
import yaml

# Create YOLO data.yaml configuration
data_config = {
    'path': DATASET_PATH,
    'train': 'images/train',
    'val': 'images/val',
    'nc': NUM_CLASSES,
    'names': CLASS_NAMES
}

config_path = os.path.join(WORK_DIR, 'data.yaml')
with open(config_path, 'w') as f:
    yaml.dump(data_config, f, sort_keys=False)

print(f"✓ Configuration saved to: {config_path}")
print("\nConfiguration:")
print(yaml.dump(data_config, sort_keys=False))

## Step 6: Verify Dataset Structure

In [ ]:
import glob

# Check train images and labels
train_images = glob.glob(os.path.join(DATASET_PATH, 'images/train/*'))
train_labels = glob.glob(os.path.join(DATASET_PATH, 'labels/train/*'))

# Check val images and labels
val_images = glob.glob(os.path.join(DATASET_PATH, 'images/val/*'))
val_labels = glob.glob(os.path.join(DATASET_PATH, 'labels/val/*'))

print("📊 Dataset Statistics:")
print(f"Train Images: {len(train_images)}")
print(f"Train Labels: {len(train_labels)}")
print(f"Val Images: {len(val_images)}")
print(f"Val Labels: {len(val_labels)}")

if len(train_images) == 0:
    print("\n❌ ERROR: No training images found!")
    print(f"Check: {os.path.join(DATASET_PATH, 'images/train/')}")
elif len(train_images) != len(train_labels):
    print(f"\n⚠️ WARNING: Mismatch between images ({len(train_images)}) and labels ({len(train_labels)})")
else:
    print("\n✓ Dataset structure looks good!")

## Step 7: Visualize Sample Images (Optional)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from PIL import Image

# Display first 3 training images
sample_images = train_images[:3]

fig, axes = plt.subplots(1, min(3, len(sample_images)), figsize=(15, 5))
if len(sample_images) == 1:
    axes = [axes]

for idx, img_path in enumerate(sample_images):
    img = Image.open(img_path)
    axes[idx].imshow(img)
    axes[idx].set_title(f"Sample {idx+1}")
    axes[idx].axis('off')

plt.tight_layout()
plt.show()
print("✓ Sample images displayed")

## Step 8: Train YOLOv8 Model

Training with **small dataset optimizations**:
- Small batch size (8)
- Fewer epochs (30-50)
- Nano model (yolov8n) for faster training
- Data augmentation enabled

In [ ]:
from ultralytics import YOLO

# Initialize YOLOv8 nano model (fastest)
model = YOLO('yolov8n.pt')

# Training parameters optimized for small dataset
results = model.train(
    data=config_path,
    epochs=30,              # Fewer epochs for demo (adjust 30-50)
    imgsz=640,              # Image size
    batch=8,                # Small batch size
    name='traffic_mini',
    project=WORK_DIR,
    cache=False,            # Don't cache (small dataset)
    workers=2,              # Fewer workers
    device=0,               # Use GPU
    patience=10,            # Early stopping patience
    save=True,
    plots=True,
    # Data augmentation (helps with small datasets)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0
)

print("\n✓ Training complete!")

## Step 9: Evaluate Model Performance

In [ ]:
# Validate the model
metrics = model.val()

print("\n📊 Model Performance:")
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

## Step 10: Test Model on Sample Image

In [ ]:
# Test on first validation image
if len(val_images) > 0:
    test_image = val_images[0]
    results = model.predict(test_image, conf=0.25)
    
    # Display results
    result_plot = results[0].plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(result_plot, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Model Prediction on Validation Image')
    plt.show()
    print("✓ Prediction displayed")
else:
    print("⚠️ No validation images available for testing")

## Step 11: Export to TFLite Format

## Step 11A: Re-export Existing Model (Standalone)

**Use this cell if you already trained the model and just need to re-export without INT8.**

This cell works independently - just run cells 1-2 to mount Drive, then run this cell.

In [ ]:
#!/usr/bin/env python3
# STANDALONE CELL - Run cells 1-2 first, then run this cell
# This will find your existing trained model and re-export as FLOAT32

import os
import glob
import shutil

print("🔍 Searching for trained models in Google Drive...")

# Search in Google Drive for the model
drive_model_paths = [
    "/content/drive/MyDrive/traffic_signs_work/traffic_mini/weights/best.pt",
    "/content/drive/MyDrive/RoadAI_Models/traffic_signs_mini_best.pt",
    "/content/drive/MyDrive/models/traffic_signs/best.pt",
]

# Also check local Colab storage
local_model_paths = [
    "/content/traffic_signs_work/traffic_mini/weights/best.pt",
    "/content/runs/detect/traffic_mini/weights/best.pt",
]

# Combine all paths
all_paths = drive_model_paths + local_model_paths

# Find existing model
model_found = None
for path in all_paths:
    if os.path.exists(path):
        model_found = path
        print(f"✅ Found model: {path}")
        break

# If not found, search more broadly
if not model_found:
    print("\n🔍 Searching all of Google Drive...")
    search_patterns = [
        "/content/drive/MyDrive/**/*best.pt",
        "/content/drive/MyDrive/**/*traffic*.pt",
    ]
    
    for pattern in search_patterns:
        files = glob.glob(pattern, recursive=True)
        if files:
            print(f"\nFound {len(files)} potential model file(s):")
            for i, f in enumerate(files):
                print(f"  {i+1}. {f}")
            
            # Use the first one
            model_found = files[0]
            print(f"\n✅ Using: {model_found}")
            break

if model_found:
    print(f"\n📦 Loading model from: {model_found}")
    
    # Install ultralytics if needed
    try:
        from ultralytics import YOLO
    except ImportError:
        print("Installing ultralytics...")
        import subprocess
        subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)
        from ultralytics import YOLO
    
    # Load model
    model = YOLO(model_found)
    print("✅ Model loaded successfully")
    
    # Export to TFLite as FLOAT32 (NO INT8)
    print("\n🔄 Exporting to TFLite format as FLOAT32 (NO INT8)...")
    print("⏳ This may take 2-3 minutes...")
    
    tflite_path = model.export(
        format='tflite',
        imgsz=640,
        # NO int8=True - exporting as float32 for better accuracy
    )
    
    print(f"\n✅ Export complete!")
    print(f"📁 TFLite file: {tflite_path}")
    
    # Check file size
    if os.path.exists(tflite_path):
        file_size = os.path.getsize(tflite_path) / (1024 * 1024)
        print(f"📏 File size: {file_size:.2f} MB")
        
        if file_size < 5:
            print("⚠️ WARNING: File is small (~3MB) - might still be INT8!")
            print("Expected size for float32: 10-13 MB")
        else:
            print("✅ File size looks correct for FLOAT32 model")
    
    # Copy to Google Drive
    drive_output_dir = "/content/drive/MyDrive/RoadAI_Models"
    os.makedirs(drive_output_dir, exist_ok=True)
    
    drive_output_path = os.path.join(drive_output_dir, "traffic_signs_mini_float32.tflite")
    
    # Find the actual exported file
    export_dir = os.path.dirname(tflite_path)
    exported_files = glob.glob(os.path.join(export_dir, "*_saved_model", "*.tflite"))
    
    if exported_files:
        # Use float32 version if available
        float32_files = [f for f in exported_files if "float32" in f or "int8" not in f]
        source_file = float32_files[0] if float32_files else exported_files[0]
        
        shutil.copy(source_file, drive_output_path)
        
        final_size = os.path.getsize(drive_output_path) / (1024 * 1024)
        
        print(f"\n✅ Model saved to Google Drive!")
        print(f"📁 Location: {drive_output_path}")
        print(f"📏 Size: {final_size:.2f} MB")
        print(f"\n📱 Next Steps:")
        print(f"   1. Download from Google Drive: RoadAI_Models/traffic_signs_mini_float32.tflite")
        print(f"   2. Rename to: traffic_signs.tflite.tflite")
        print(f"   3. Replace in: app/src/main/assets/models/")
        print(f"   4. Rebuild app: .\\gradlew assembleDebug")
        
        if final_size > 8:
            print(f"\n✅ SUCCESS: Float32 export confirmed ({final_size:.1f}MB)")
        else:
            print(f"\n⚠️ File might still be quantized. Try re-running the cell.")
    else:
        print(f"\n⚠️ Could not find exported TFLite file")
        print(f"Searched in: {export_dir}")
        
else:
    print("\n❌ ERROR: No trained model found!")
    print("\n📝 You need to either:")
    print("   1. Run cells 1-10 to train a new model, OR")
    print("   2. Upload your trained best.pt file to Google Drive")
    print("      Suggested location: /content/drive/MyDrive/RoadAI_Models/")
    print("\n💡 TIP: If you trained the model before, check your Google Drive")
    print("   for any .pt files in the traffic_signs_work folder")

In [ ]:
# Find the best model weights
import os
import glob
from ultralytics import YOLO

WORK_DIR = "/content/traffic_signs_work"
best_model_path = os.path.join(WORK_DIR, 'traffic_mini', 'weights', 'best.pt')

if os.path.exists(best_model_path):
    # Load best model
    best_model = YOLO(best_model_path)
    
    # Export to TFLite WITHOUT int8 quantization (float32)
    print("Exporting to TFLite format as FLOAT32...")
    tflite_path = best_model.export(
        format='tflite',
        imgsz=640,
        # int8=True removed - using float32 for better compatibility
    )
    print(f"✓ TFLite model exported: {tflite_path}")
    print(f"✓ Model type: FLOAT32 (better accuracy and compatibility)")
else:
    print(f"❌ Best model not found at: {best_model_path}")
    print(f"\nTrying to find any trained model in {WORK_DIR}...")
    
    # Search for any .pt model files
    model_files = glob.glob(os.path.join(WORK_DIR, '**', '*.pt'), recursive=True)
    if len(model_files) > 0:
        print(f"\nFound {len(model_files)} model file(s):")
        for i, mf in enumerate(model_files):
            print(f"  {i+1}. {mf}")
        
        # Use the first found model
        print(f"\nUsing: {model_files[0]}")
        best_model = YOLO(model_files[0])
        
        tflite_path = best_model.export(
            format='tflite',
            imgsz=640,
            # int8=True removed
        )
        print(f"✓ TFLite model exported: {tflite_path}")
    else:
        print("\n❌ No trained models found!")
        print("You need to run cell 8 first to train the model.")

## Step 12: Copy Model to Google Drive

In [ ]:
import shutil

# Define output path in Google Drive
drive_output_path = "/content/drive/MyDrive/RoadAI_Models/traffic_signs_mini.tflite"

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(drive_output_path), exist_ok=True)

# Find the exported TFLite file
tflite_files = glob.glob(os.path.join(WORK_DIR, 'traffic_mini', 'weights', '*_saved_model', '*_int8.tflite'))

if len(tflite_files) > 0:
    source_tflite = tflite_files[0]
    shutil.copy(source_tflite, drive_output_path)
    
    # Get file size
    file_size = os.path.getsize(drive_output_path) / (1024 * 1024)
    
    print(f"✓ Model saved to Google Drive!")
    print(f"Location: {drive_output_path}")
    print(f"Size: {file_size:.2f} MB")
    print(f"\n📱 Next Steps:")
    print(f"1. Download this file from Google Drive")
    print(f"2. Rename it to: traffic_signs.tflite.tflite")
    print(f"3. Place in: app/src/main/assets/models/")
    print(f"4. Update TrafficSignDetector.kt numClasses to: {NUM_CLASSES}")
else:
    print("❌ TFLite file not found!")
    print(f"Search path: {os.path.join(WORK_DIR, 'traffic_mini', 'weights')}")

## Step 13: Display Training Results

In [ ]:
# Display training curves
results_img = os.path.join(WORK_DIR, 'traffic_mini', 'results.png')

if os.path.exists(results_img):
    img = Image.open(results_img)
    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Results')
    plt.show()
    print("✓ Training curves displayed")
else:
    print("⚠️ Results image not found")

## 📝 Summary

### What This Notebook Does:
1. ✓ Mounts Google Drive
2. ✓ Loads your custom mini dataset
3. ✓ Trains YOLOv8 model (optimized for small data)
4. ✓ Evaluates performance
5. ✓ Exports to TFLite format
6. ✓ Saves model to Google Drive

### Important Notes:
- **Dataset Size**: Recommended 20-50 images for quick demo
- **Training Time**: ~10-20 minutes with small dataset
- **Accuracy**: Limited due to small dataset (good for demo only)
- **Production Use**: Train with larger dataset (500+ images) for real deployment

### Model Location:
- **Google Drive**: `/content/drive/MyDrive/RoadAI_Models/traffic_signs_mini.tflite`
- **Download and rename**: `traffic_signs.tflite.tflite`
- **Place in app**: `app/src/main/assets/models/`